In [0]:
%pip install seaborn
%pip install xgboost

In [0]:
import mlflow, os
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models.signature import infer_signature

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from config import DATASET_DIR, VOLUME_FEATURES_DIR, EXPERIMENT_NAME, get_run_id

In [0]:
parent_run_id = get_run_id()

In [0]:
X_train_optimal = np.load(f"{VOLUME_FEATURES_DIR}/X_train_optimal.npy")
y_train_labels = np.load(f"{VOLUME_FEATURES_DIR}/y_train_labels.npy")
X_val_optimal = np.load(f"{VOLUME_FEATURES_DIR}/X_val_optimal.npy")
y_val_labels = np.load(f"{VOLUME_FEATURES_DIR}/y_val_labels.npy")
class_names = sorted([d for d in os.listdir(DATASET_DIR) if not d.startswith(".")])

In [0]:
def evaluate_and_log_model(
    model,
    model_name: str,
    params: dict,
    X_train,
    y_train,
    X_val,
    y_val,
    class_names: list
):
    
    mlflow.set_experiment(EXPERIMENT_NAME)

    with mlflow.start_run(run_name = model_name, nested = True) as run:

        mlflow.log_params(params)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)

        val_acc = accuracy_score(y_val, y_pred)
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            y_val, y_pred, average="macro", zero_division=0
        )

        metrics = {
            "val_accuracy": val_acc,
            "val_f1_macro": f1_macro,
            "val_precision_macro": precision_macro,
            "val_recall_macro": recall_macro,
        }

        _, _, per_class_f1, _ = precision_recall_fscore_support(
            y_val, y_pred, average=None, zero_division=0
        )
        for idx, c_name in enumerate(class_names):
            metrics[f"f1_class_{c_name}"] = per_class_f1[idx]

        mlflow.log_metrics(metrics)


        cm = confusion_matrix(y_val, y_pred)
        cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            cm_normalized,
            annot=True,
            fmt=".2f",
            cmap="Blues", 
            xticklabels=class_names,
            yticklabels=class_names,
            ax=ax
        )
        ax.set_title(f"Confusion Matrix - {model_name}")
        ax.set_ylabel("True Diagnosis")
        ax.set_xlabel("Predicted Diagnosis")
        plt.tight_layout()

        cm_filepath = f"{model_name}_confusion_matrix.png"
        fig.savefig(cm_filepath)
        plt.close(fig)

        mlflow.log_artifact(cm_filepath, artifact_path="diagnostic_plots")

        
        report = classification_report(y_val, y_pred, target_names=class_names)
        with open(f"{model_name}_classification_report.txt", "w") as f:
            f.write(report)
        mlflow.log_artifact(f"{model_name}_classification_report.txt", artifact_path="reports")

        
        signature = infer_signature(X_train[:5], model.predict(X_train[:5]))

        
        if "xgb" in model_name.lower():
            mlflow.xgboost.log_model(model, artifact_path="model", signature=signature)
        else:
            mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)

        print(f"Run completed: {model_name} | Val Macro F1: {f1_macro:.4f} | Val Accuracy: {val_acc:.4f}")
        return run.info.run_id




In [0]:

lr_params = {
    "C": 0.1,                          
    "penalty": "l2",                  
    "solver": "lbfgs",                 
    "max_iter": 1000,
    "multi_class": "multinomial",
    "random_state": 14
}

lr_model = LogisticRegression(
    C=lr_params["C"],
    penalty=lr_params["penalty"],
    solver=lr_params["solver"],
    max_iter=lr_params["max_iter"],
    multi_class=lr_params["multi_class"],
    random_state=lr_params["random_state"]
)

In [0]:


rf_params = {
    "n_estimators": 150,               
    "max_depth": 12,                   
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "criterion": "gini",               
    "random_state": 14,
    "n_jobs": -1
}

rf_model = RandomForestClassifier(
    n_estimators=rf_params["n_estimators"],
    max_depth=rf_params["max_depth"],
    min_samples_split=rf_params["min_samples_split"],
    min_samples_leaf=rf_params["min_samples_leaf"],
    criterion=rf_params["criterion"],
    random_state=rf_params["random_state"],
    n_jobs=rf_params["n_jobs"]
)

In [0]:


xgb_params = {
    "n_estimators": 200,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "multi:softprob",
    "num_class": len(class_names),
    "eval_metric": "mlogloss",
    "random_state": 14
}

xgb_model = XGBClassifier(
    n_estimators=xgb_params["n_estimators"],
    max_depth=xgb_params["max_depth"],
    learning_rate=xgb_params["learning_rate"],
    subsample=xgb_params["subsample"],
    colsample_bytree=xgb_params["colsample_bytree"],
    objective=xgb_params["objective"],
    num_class=xgb_params["num_class"],
    eval_metric=xgb_params["eval_metric"],
    random_state=xgb_params["random_state"],
    n_jobs=-1
)

In [0]:
with mlflow.start_run(run_id = parent_run_id):

    lr_run_id = evaluate_and_log_model(
    model=lr_model,
    model_name="ResNet50_Logistic_Regression",
    params=lr_params,
    X_train=X_train_optimal,
    y_train=y_train_labels,
    X_val=X_val_optimal,
    y_val=y_val_labels,
    class_names=class_names
    )

    rf_run_id = evaluate_and_log_model(
    model=rf_model,
    model_name="ResNet50_Random_Forest",
    params=rf_params,
    X_train=X_train_optimal,
    y_train=y_train_labels,
    X_val=X_val_optimal,
    y_val=y_val_labels,
    class_names=class_names
    )

    xgb_run_id = evaluate_and_log_model(
    model=xgb_model,
    model_name="ResNet50_XGBoost",
    params=xgb_params,
    X_train=X_train_optimal,
    y_train=y_train_labels,
    X_val=X_val_optimal,
    y_val=y_val_labels,
    class_names=class_names
    )


    

In [0]:
from mlflow import MlflowClient

client = MlflowClient()
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run_id}'",
    order_by=["metrics.val_f1_macro DESC"]
)

leaderboard = runs_df[[
    "tags.mlflow.runName",
    "metrics.val_f1_macro",
    "metrics.val_accuracy",
    "metrics.val_precision_macro",
    "metrics.val_recall_macro"
]]
print("--- Model Experiment Leaderboard ---")
print(leaderboard.to_string(index=False))


best_run = runs_df.iloc[0]
best_run_id = best_run["run_id"]
best_run_name = best_run["tags.mlflow.runName"]
best_macro_f1 = best_run["metrics.val_f1_macro"]

print(f"\nWinning Model: {best_run_name} (Run ID: {best_run_id}) with Val Macro F1 = {best_macro_f1:.4f}")